# Multi-Laminar Cortical Modeling and Stable Fine-Tuning with AGSDR

Package-native jaxfne workflow: configure, edit, construct, simulate, stimulate a selected subset, tune with AGSDR, visualize, and export.

Scope: `truth_safe_unverified`, `computational_scaffold`, `laminar_proxy_no_pde`, `physical_amplitude_claim_allowed=False`.

## Install: PyPI stable

In [ ]:
!pip install -q jaxfne

## Install: GitHub dev branch

Run this after the PyPI cell when using the latest development branch.

In [ ]:
!pip install -q "jaxfne @ git+https://github.com/HNXJ/jaxfne.git@dev"

## Imports and output directories

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

import jaxfne as jtfne

Continue the package-native workflow.

In [ ]:
OUTPUT_DIR = Path("outputs/multi_laminar_cortical_agsdr")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

Continue the package-native workflow.

In [ ]:
print("jaxfne", jtfne.__version__)
print(jtfne.runtime_report(jtfne.RuntimeConfig(dtype="float32", jit=True)))

## 1. Get a cortex configuration

In [ ]:
cfg0 = jtfne.default_spectrolaminar_config(
    areas=["V1", "V4"], n_per_area=80,
    seed=20260530, duration_ms=1000.0, dt_ms=0.1,
)
pd.Series(jtfne.configuration_table(cfg0))

## 2. Edit every major configuration domain

Several assignments intentionally repeat default-like values to show editability.

In [ ]:
cfg = cfg0.runtime(
    seed=20260530, duration_ms=1000.0, dt_ms=0.1,
    dtype="float32", jit=True, vmap=False,
)

Continue the package-native workflow.

In [ ]:
cfg = cfg.drive(
    baseline_drive_by_cell_type={"E": 4.0, "PV": 2.0, "SST": 2.2, "VIP": 2.0},
    drive_by_layer={"L4": 1.0}, drive_by_area={"V1": 1.0, "V4": 1.0},
    time_schedule="constant_plus_event", evoked_windows=[(250.0, 400.0)],
    noise_policy="additive_poisson", trial_variability=True,
)

Continue the package-native workflow.

In [ ]:
cfg = cfg.inter_column_connectivity(
    source_area="V1", target_area="V4", mode="sparse",
    p_feedforward=0.06, p_feedback=0.055,
    feedforward_weight_range=(0.007, 0.030),
    feedback_weight_range=(0.006, 0.026),
    delay_ms_or_status="proxy_no_delay_solver", seed=20260530,
)

Continue the package-native workflow.

In [ ]:
cfg = cfg.field(
    domain="laminar_column", source_projection_mode="proxy_no_field_solve",
    field_solver_status="laminar_proxy_no_pde", conductivity_status="declared_proxy",
    boundary_condition="mean_zero_neumann_metadata", gauge="mean_zero_metadata",
    physical_amplitude_claim_allowed=False,
)

Continue the package-native workflow.

In [ ]:
cfg = cfg.probes(
    ["spikes", "V_m", "source", "LFP", "CSD", "EEG", "MEG", "EMM"],
    n_contacts=32, name="multi_laminar_proxy_probe",
    units_or_status="proxy_units",
)

Continue the package-native workflow.

In [ ]:
cfg = cfg.objective(
    firing_rate_target={"overall": 5.0},
    band_definitions={"alpha_beta": (10.0, 25.0), "gamma": (40.0, 150.0)},
    synchrony_metrics=["kappa_synchrony"],
    rejection_gates={"physical_amplitude_claim_allowed": (False, False)},
)

Continue the package-native workflow.

In [ ]:
cfg = cfg.optimizer(
    optimizer_family="AGSDR", budget=24,
    search_space={"noise_amplitude": (0.0, 2.0), "drive_scale": (0.5, 1.5),
                  "local_exc_gain": (0.8, 1.2), "local_inh_gain": (0.8, 1.2)},
    hard_gates={"claim_level": "computational_scaffold"},
)

Continue the package-native workflow.

In [ ]:
validation = jtfne.validate_configuration(cfg)
print(validation)
pd.Series(jtfne.configuration_table(cfg))

## 3. Construct from the edited configuration

In [ ]:
model = jtfne.construct(cfg)
model_summary = model.summary()
model_summary

## 4. Simulate baseline activity

In [ ]:
sim = jtfne.Simulation(
    duration_ms=1000.0, dt_ms=0.1, seed=20260530,
    record_sources=True, record_fields=True,
    runtime=jtfne.RuntimeConfig(dtype="float32", jit=True),
)

Continue the package-native workflow.

In [ ]:
signals_baseline = model.simulate(sim)
readout_baseline = model.probe(
    signals_baseline, modes=["spikes", "V_m", "source", "LFP", "CSD"]
)

Continue the package-native workflow.

In [ ]:
baseline_summary = signals_baseline.summary()
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, dt_ms=sim.dt_ms)
print("baseline summary", baseline_summary)
print("baseline kappa", baseline_kappa)

## 5. Stimulate a custom selected subset

In [ ]:
target_indices = jtfne.select_neurons(model, area="V1", layer="L4", cell_type="E")
print("selected V1/L4/E proxy target count:", len(target_indices))
print("first selected indices:", target_indices[:10])

Continue the package-native workflow.

In [ ]:
stim_event = {
    "label": "custom_V1_L4_E_drive", "onset_ms": 250.0,
    "duration_ms": 150.0, "amplitude": 1.25,
    "target_indices": target_indices,
}
stim = jtfne.stimulus_schedule([stim_event], n_neurons=model_summary["n_units"])

Continue the package-native workflow.

In [ ]:
signals_stim = model.simulate(sim, paradigm=stim)
stim_summary = signals_stim.summary()
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, dt_ms=sim.dt_ms)
print("stimulated summary", stim_summary)
print("stimulated kappa", stim_kappa)

## 6. AGSDR fine tuning toward 5 Hz and kappa synchrony 0.0

In [ ]:
objective = jtfne.rate_synchrony_targets(
    target_rate_hz=5.0, target_kappa_synchrony=0.0,
    rate_weight=1.0, synchrony_weight=0.25,
)

Continue the package-native workflow.

In [ ]:
optimizer = jtfne.agsdr(
    parameters={"noise_amplitude": (0.0, 2.0), "drive_scale": (0.5, 1.5),
                "local_exc_gain": (0.8, 1.2), "local_inh_gain": (0.8, 1.2)},
    generations=6, population_size=6, seed=20260530,
)

Continue the package-native workflow.

In [ ]:
tune_sim = jtfne.Simulation(
    duration_ms=500.0, dt_ms=0.1, seed=20260530,
    record_sources=True, record_fields=True,
    runtime=jtfne.RuntimeConfig(dtype="float32", jit=True),
)

Continue the package-native workflow.

In [ ]:
tuned = model.tune(objectives=objective, optimizer=optimizer, simulation=tune_sim, seed=20260530)
tuned.summary

## 7. Simulate tuned model and compare summaries

In [ ]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_summary = signals_tuned.summary()
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, dt_ms=sim.dt_ms)

Continue the package-native workflow.

In [ ]:
summary_table = pd.DataFrame([
    {"condition": "baseline", "rate_hz": baseline_summary["spike_rate_hz_mean"], "kappa": baseline_kappa},
    {"condition": "stimulated", "rate_hz": stim_summary["spike_rate_hz_mean"], "kappa": stim_kappa},
    {"condition": "tuned+stimulated", "rate_hz": tuned_summary["spike_rate_hz_mean"], "kappa": tuned_kappa},
])
summary_table

## 8. Spectrolaminar suite with custom frequency limits and resolution

In [ ]:
fig = jtfne.vis.spectrolaminar_suite(
    signals_tuned, freq_min_hz=1.0, freq_max_hz=150.0,
    freq_count=128, psd_nperseg=512, figsize=(14, 10), dpi=160,
    title="Multi-laminar cortical AGSDR tuning - simulated proxy readouts",
)

Continue the package-native workflow.

In [ ]:
fig_path = FIG_DIR / "multi_laminar_spectrolaminar_suite.png"
fig.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print(fig_path)

## 9. Export manifest and validation report

In [ ]:
evaluation_report = tuned_model.evaluate(signals_tuned, objective)
readout_tuned = tuned_model.probe(
    signals_tuned, modes=["spikes", "V_m", "source", "LFP", "CSD"]
)

Continue the package-native workflow.

In [ ]:
manifest = jtfne.manifest(
    cfg, signals=signals_tuned, readout=readout_tuned,
    runtime_config=sim.runtime, paradigm=stim.to_dict(),
    objective={"name": objective.name, "kind": objective.kind},
    evaluation=evaluation_report, tuning=tuned.summary,
)

Continue the package-native workflow.

In [ ]:
validation_report = {
    "tutorial_id": "multi_laminar_cortical_agsdr",
    "finite_outputs": True, "json_safe": True,
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_claim_allowed": False,
}

Continue the package-native workflow.

In [ ]:
validation_report["target_rate_hz"] = 5.0
validation_report["target_kappa_synchrony"] = 0.0
validation_report["achieved_rate_hz"] = evaluation_report.get("achieved_rate_hz")
validation_report["achieved_kappa_synchrony"] = evaluation_report.get("achieved_kappa_synchrony")
validation_report["figure_paths"] = [str(fig_path)]

Continue the package-native workflow.

In [ ]:
jtfne.save_json(manifest, OUTPUT_DIR / "manifest.json")
jtfne.save_json(validation_report, OUTPUT_DIR / "validation_report.json")
json.dumps(manifest, allow_nan=False)
json.dumps(validation_report, allow_nan=False)
print("wrote", OUTPUT_DIR.resolve())

## Interpretation

Use the summary table and spectrolaminar panel to inspect how selected-subset drive and AGSDR-selected parameters change overall rate and kappa synchrony. The readouts are simulated/proxy readouts for relative within-run comparison.

## Failure modes

- Empty target subset: broaden the selector or increase `n_per_area`.
- High kappa after tuning: increase synchrony weight or reduce recurrent gain bounds.
- Silent runs: increase drive or lower inhibitory gain.
- Long Colab runtime: reduce population or AGSDR budget for smoke execution.

## Exercises

1. Change the target subset from V1/L4/E to V4/L2/3/PV.
2. Tune to 8 Hz.
3. Increase `synchrony_weight` to 1.0.
4. Set `freq_max_hz=80.0`.
5. Add a second targeted event.

## Scope boundary

This notebook covers package-native configuration, simulation, custom native-drive stimulation, black-box AGSDR tuning, and proxy spectrolaminar visualization. It keeps `truth_safe_unverified`, `computational_scaffold`, `laminar_proxy_no_pde`, and `physical_amplitude_claim_allowed=False`. It does not provide calibrated LFP/CSD amplitudes, real EEG/MEG forward modeling, biological metabolism, mechanism proof, or a solved PDE field model.